# All Table 1 rows trained with MolE's own recipe (GPU)

MolE's published 0.7797 comes from one configuration: `n_estimators=1000, max_depth=100, eta=0.1, subsample=1.0`, raw features, no early stopping, no class weighting. Locally this reproduced MolE at test 0.7802 with `tree_method=exact`. This notebook trains **every** row (MolE, RDKit, MolE+RDKit, MolE + each GPT set) with that same recipe, so the rows are compared on MolE's terms.

- The GPU supports `hist` only, not `exact`. With `hist` on CPU, MolE alone scored 0.7655, so expect all rows to sit about 0.015 lower than an `exact` run. Rows are still comparable with each other, because they all use the same method.
- The configuration is deterministic (subsample = 1), so seeds add no spread. Confidence intervals come from resampling the 120 test compounds (cell 5).
- Optional cell 6 runs the true `exact` method on Kaggle's CPU (about 1 h per row, so roughly 9-10 h); use it only if you need numbers on MolE's exact scale.

**Setup:** upload the new `gpt_mole_bundle.zip` as a new version of your dataset, attach it, set the accelerator to GPU, then Run All (or Save & Run All).

In [ ]:
# 1. Confirm XGBoost really trains on the GPU. XGBoost silently falls back to CPU
#    when no GPU is visible, so check for that warning rather than trusting the setting.
import subprocess, sys, warnings
import numpy as np
import xgboost as xgb

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| xgboost", xgb.__version__)
assert int(xgb.__version__.split(".")[0]) >= 2, "device='cuda' needs xgboost >= 2"

X = np.random.rand(2000, 20)
y = (X[:, 0] > 0.5).astype(int)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    xgb.train({"tree_method": "hist", "device": "cuda", "objective": "binary:logistic"},
              xgb.DMatrix(X, label=y), 10)
fallback = [str(w.message) for w in caught if "GPU" in str(w.message)]
assert not fallback, f"XGBoost fell back to CPU - turn on a GPU accelerator.\n{fallback}"
print("GPU training OK")

In [ ]:
# 2. Dependencies. build_train.py imports rdkit (unused by these runs) and reads the
#    SF3 workbook with openpyxl; install either only if the image lacks it.
import importlib.util

for module, package in [("rdkit", "rdkit"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import pandas, sklearn
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
# 3. Copy the bundle to a writable folder (inputs are read-only) and work from there.
#    Also restores finished runs from a previous version's output, if one is attached.
import glob, os, shutil

hits = glob.glob("/kaggle/input/**/notebooks/build_train.py", recursive=True)
assert hits, "Bundle not found - attach the dataset made from gpt_mole_bundle.zip."
BUNDLE = os.path.dirname(os.path.dirname(hits[0]))
WORK = "/kaggle/working/gpt_mole"
if not os.path.exists(WORK):
    shutil.copytree(BUNDLE, WORK)
os.chdir(WORK)
print("bundle:", BUNDLE, "-> working in", os.getcwd())
if os.path.exists("MANIFEST.txt"):
    print(open("MANIFEST.txt").read())

# Kaggle decompresses .gz files inside an uploaded dataset, so x.tsv.gz arrives as
# x.tsv. Re-compress them to the names the scripts expect.
import gzip
sys.path.insert(0, "scripts")
import run_table1
needed = [run_table1.SF3, run_table1.SPLIT, run_table1.MOLE, run_table1.PRESTWICK,
          run_table1.CHEMDESC, *run_table1.FREEFORM.values()]
for path in needed:
    if path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path[:-3]):
        with open(path[:-3], "rb") as src, gzip.open(path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("re-compressed", path)
missing = [p for p in needed if not os.path.exists(p)]
assert not missing, f"Missing inputs: {missing}"
print(f"all {len(needed)} inputs present")

OUT_ROOT = "results/mole_method_all_rows"
previous = [d for d in glob.glob("/kaggle/input/**/mole_method_all_rows", recursive=True)
            if not d.startswith(BUNDLE)]
for d in previous:
    shutil.copytree(d, OUT_ROOT, dirs_exist_ok=True)
    print("restored finished runs from", d)

In [ ]:
# 4. Train all 9 rows on the GPU with MolE's recipe (hist). Finished rows are skipped.
!python -u scripts/run_mole_method.py --tree_method hist --device cuda --output_dir {OUT_ROOT}
assert _exit_code == 0, "a row failed - see the output above"

In [ ]:
# 5. Table: validation, test, 95% bootstrap CI, and paired difference vs MolE alone.
import pandas as pd
pd.set_option("display.width", 240)
pd.read_csv(f"{OUT_ROOT}/table_mole_method_hist_cuda.csv")

In [ ]:
# 6. OPTIONAL, slow: MolE-exact method on Kaggle's CPU (~1 h per row). Comment out to skip.
# !python -u scripts/run_mole_method.py --tree_method exact --device cpu --output_dir {OUT_ROOT}
# pd.read_csv(f"{OUT_ROOT}/table_mole_method_exact.csv")

In [ ]:
# 7. Package results for download (Output tab -> mole_method_results.zip).
archive = shutil.make_archive("/kaggle/working/mole_method_results", "zip", OUT_ROOT)
print(archive, f"{os.path.getsize(archive) / 1024 ** 2:.1f} MB")